In [ ]:
from typing import Iterable
from itertools import chain as iterchain, combinations as itercomb
from collections import Counter
from pprint import pprint

iterflat = iterchain.from_iterable

In [ ]:
from board import DIGITS, Digits, Loc, Cell, Board

# Topology

The board is classicaly 9 rows, 9 columns, and 9 boxes over them. These are major units.

Intersection of a row and a column gives individual cell. A cell is subdivided further into 9 segments for digits, but only for visualizing purposes.

Intersection of a box with a row or column gives sector.

Generalized, all the localities can be addressed by a set of `(box, col, row)`

Visibility of 2 localities means they share some unit (or two). It determines location of cell peers and possibly conflicting drafts.


In [ ]:
from topology import Zone, visibility, allvisible

In [ ]:
# construction
assert Zone.B(1) == Zone(1, ..., ...)
assert Zone.R(2) == Zone(..., 2, ...)
assert Zone.C(3) == Zone(..., ..., 3)
assert Zone.L(Loc(2, 3)) == Zone(1, 2, 3)  # fulfiled box

assert Zone(1, 2, 3).is_cell
assert Zone(1, ..., ...).is_unit
assert Zone(1, 2, ...).is_sector

# intersection
assert Zone.R(2) & Zone.C(3) == Zone(1, 2, 3)
assert Zone.B(1) & Zone.R(2) == Zone(1, 2, ...)
assert Zone.B(1) & Zone.C(3) == Zone(1, ..., 3)
# containering
assert set(iter(Zone(1, 2, ...))) == {Loc(2, 1), Loc(2, 2), Loc(2, 3)}
assert Loc(2, 3) in Zone.B(1)
assert Loc(2, 3) in Zone.R(2)
assert Loc(2, 3) in Zone.C(3)

In [ ]:
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(2, 3))) == {Zone.B(1)}
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(1, 3))) == {Zone.R(1), Zone.B(1)}
assert visibility(Zone(1, 2, ...), Zone.L(Loc(2, 9))) == {Zone.R(2)}
assert visibility(Zone.L(Loc(1, 2)), Zone.L(Loc(8, 9))) == set()

assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(8, 9))) == {Zone(3, 1, 9), Zone(7, 8, 2)}
assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(2, 3))) == {Zone(1, ..., ...)}
assert allvisible(Zone.L(Loc(1, 2)), Zone.L(Loc(3, 2))) == {Zone(1, ..., ...), Zone(..., ..., 2)}

In [ ]:
print(*map(str, Zone.Units()))

# Core Logic

The core rueles of Sudoku:

- all cells in every unit contain different digits
- every digit is contained only once in every unit

The core techniques:

- if a digit is placed in a cell, clear drafts of the digit in all peer cells
- if a draft of digit is unique in a unit, place the digit in the cell, and go to the naked state

Logic of the core:

- a draft is a statement $A(l_A, d_A)$ assuming digit $d_A$ should be placed at location $l_A$
- conflicting statements would be $B(l_B \in Peers(A), d_A)$ and $B(l_A, d_B \ne d_A)$
- with $Peers_U(A) = \{ l : l \in U, l \neq l_A \}$ — visible peers of the cell $l_A$ in a unit $U$
- (elimination) $A = \mathcal{T} \implies \forall U \ni l_A: \forall B(l_B \in Peers_U(A), d_A) = \mathcal{F}$
    - (in all units containing the location, all matching peer-drafts are invalid)
- (finalization) for some $A|l_A \in U, A = \mathcal{T} \Longleftarrow \forall B(l_B \in Peers_U(A), d_A) = \mathcal{F}$
    - (when all matching peer drafts in some unit are already invalid)
- (propagation) $(A = \mathcal{T})|_{some~unit} \equiv (A = \mathcal{T})|_{all~units}$
    - (obviously)

The elimination logic corresponds to 'naked single' and finalization corresponds to 'hidden single'.


#### Notation

- OR: $A \vee B$
- XOR: $A \veebar B$
- AND: $A \wedge B$, also $(blabla) \cdot (blabla)$
- NAND: $A \barwedge B$
- serial OR: $\bigvee A_i$
- serial XOR: $\underline\bigvee A_i$
- serial AND: $\bigwedge A_i$
- serial NAND: $\bar\bigwedge A_i$


### Ultimate Draft Adressing

- location: cell, unit, sector
- digits: one or more digits


In [ ]:
from analysis import Node, Group

In [ ]:
assert Node.C(Cell(Loc(1, 2), Digits({1, 2, 3}))) == Node(Zone.L(Loc(1, 2)), Digits({1, 2, 3}))
assert Node.at(Loc(1, 2), Digits({1, 2, 3})) == Node(Zone.L(Loc(1, 2)), Digits({1, 2, 3}))
assert Node.at(Loc(1, 2), 5) == Node(Zone.L(Loc(1, 2)), Digits({5}))

assert Node(Zone.L(Loc(1, 2)), Digits({1})).is_cellular
assert Node(Zone.L(Loc(1, 2)), Digits({1})).is_singular
assert not Node(Zone.B(1), Digits({1})).is_cellular
assert not Node(Zone.B(1), Digits({1, 2, 3})).is_cellular

# Solving by rules

searching for patterns -> generating resolutions -> applying resolutions


In [ ]:
from utils import filt_finals, filt_having, draftborhood, draftboard, flat_cells, count_finals
from solving import Resolving, Resolver, Resolution, solver, solve_silent, solve_logging

## Singles

A single is a draft about some digit with no concurrent drafts about the same digit in a unit.

A final is a single inside a cell == solution for the cell

Rules:

- (open singles) remove all draft conflicting with finals in all units
- (hidden singles) remove all other drafts of the same digit from other units


In [ ]:
def open_singles(board: Board) -> Resolving:
    """Open singles (finals): removing drafts conflicting with finals"""

    for fincell in filter(filt_finals, iter(board)):
        findig: int = fincell.final  # type: ignore
        for zone in Zone.around(Zone.of(fincell)):
            spoilers = tuple(filter(lambda c: c != fincell and findig in c, draftborhood(board, zone)))
            if len(spoilers):
                yield Resolution(
                    castaways={Cell(c.loc, Digits({findig})) for c in spoilers},
                    highlights={
                        "zone": zone,
                        "anchors": {Node.at(fincell, findig)},
                    },
                )

In [ ]:
def hidden_singles(board: Board) -> Resolving:
    """Hidden finals: removing all other drafts from the cell of a single"""

    for zone in Zone.Units():
        drafts = tuple(draftborhood(board, zone))
        counts = Counter(flat_cells(drafts))
        for dig, cnt in counts.items():
            final = Digits({dig})
            if cnt == 1:
                (lonesome,) = filter(filt_having(dig), drafts)
                yield Resolution(
                    finals={Cell(lonesome.loc, final)},
                    highlights={
                        "zone": zone,
                        "anchors": {Node.at(lonesome, final)},
                        "empties": {Node(zone, final)},
                    },
                )

## Multiples

Combos of N digits


#### open

Some n cells (within locality) contains only n-combo // the digits may be in other cells

Rule: remove the digits of the combo from all other cells


In [ ]:
def open_mults(board: Board, mult: int) -> Resolving:
    """Clean out spoiled neighbours of open multiples in each zone"""
    for zone in Zone.Units():
        drafts = set(draftborhood(board, zone))
        inhabitants = set(flat_cells(drafts))
        for cmb in itercomb(inhabitants, mult):
            combo = Digits(cmb)
            # all cells containing only the combo
            habitat = set(filter(lambda c: c.digits <= combo, drafts))
            # all other neighbors containing some combo digits
            spoilers = tuple(filter(lambda c: c.digits & combo, drafts - habitat))

            if len(habitat) == mult and len(spoilers):
                yield Resolution(
                    castaways={Cell(c.loc, Digits(c.digits & combo)) for c in spoilers},
                    highlights={
                        "zone": zone,
                        "anchors": {Node.at(c, Digits(c.digits & combo)) for c in habitat},
                    },
                )


def open_mults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return open_mults(board, mult)

    resolver.__name__ = f"open_mults[{mult}]"
    return resolver

### hidden

Some n-combo contained in only n cells (within a unit) // along other drafts

Rule: remove all other drafts from the cells => it becomes open


In [ ]:
def hidden_mults(board: Board, mult: int) -> Resolving:
    """Clean up cellmates of hidden multiples"""
    for zone in Zone.Units():
        drafts = set(draftborhood(board, zone))
        inhabitants = set(flat_cells(drafts))
        for cmb in itercomb(inhabitants, mult):
            combo = Digits(cmb)
            if len(inhabitants & combo) != mult:
                continue
            # all cells containing some combo digits (+ some spoilers)
            habitat = tuple(filter(lambda c: c.digits & combo, drafts))
            # inhabited cells with other digits
            spoiled = tuple(filter(lambda c: c.digits - combo, habitat))
            if len(habitat) == mult and len(spoiled):
                yield Resolution(
                    castaways={Cell(c.loc, Digits(c.digits - combo)) for c in spoiled},
                    highlights={
                        "zone": zone,
                        "anchors": set(Node.at(c, c.digits & combo) for c in habitat),
                        "empties": {Node(zone, combo)},
                    },
                )


def hidden_mults_(mult: int):
    def resolver(board: Board) -> Resolving:
        return hidden_mults(board, mult)

    resolver.__name__ = f"hidden_mults[{mult}]"
    return resolver

## Links

Links represent XOR or NAND relations between drafts.

- XOR $\veebar$ corresponds to "each digit appears only once in a locality"
- NAND $\barwedge$ corrsponds to "each locality contains only different digits"

(or vise versa, I dunno)


In [ ]:
from analysis import Link, HLink, SLink

#### strong/hard links

Represent XOR relation $\veebar$

Criteria:

- only 2 drafts of same digit in a locality
- only 2 drafts (of different digits) in a cell


In [ ]:
def search_hard_segms(board: Board) -> Iterable[HLink]:
    """Search for all intra-cellular links (open pairs)"""
    for cell in draftboard(board):
        if len(cell) == 2:
            d1, d2 = cell.digits
            yield HLink((Node.at(cell, d1), Node.at(cell, d2)))


def search_hard_cells(board: Board) -> Iterable[HLink]:
    """Search for all inter-cellular links (same-digit)"""
    for zone in Zone.Units():
        drafts = tuple(draftborhood(board, zone))
        counts = Counter(flat_cells(drafts))
        for dig, cnt in counts.items():
            if cnt == 2:
                (n1, n2) = filter(filt_having(dig), drafts)
                yield HLink((Node.at(n1, dig), Node.at(n2, dig)))

#### weak/soft links

Represent NAND relation $\barwedge$

Criteria:

- any 2 drafts of same digit in a locality
- any 2 drafts (of different digits) in a cell

Note: The criteria are totally independent of board content (calculating from locations only)

Visibility = soft-linkability


In [ ]:
def check_soft(n1: Node, n2: Node):
    assert n1 != n2
    if n1.dig != n2.dig:
        # different digits within a cell
        return n1.is_cellular and n2.is_cellular and n1.zone == n2.zone
    else:
        # same digits in some shared zone
        return len(set(visibility(n1.zone, n2.zone))) > 0

## Chains

Alterating link chains constituted of `~ hard ~ soft ~` and `~ soft ~ hard ~`

Lemma1: $(X \barwedge A) \cdot (A \veebar B) \cdot (B \barwedge X) \Rightarrow \neg X$

Meaning: all draft visible (soft-linkable) from some XORed points, are all invalid

Lemma2: $(X \veebar A) \cdot (A \barwedge B) \cdot (B \veebar Y) \Rightarrow (X \veebar Y)$

Meaning: a ALC (of any length) with hard edges behaves as if its edges are hard-linked


In [ ]:
from analysis import Chain

In [ ]:
def search_soft(board: Board, n1: Node, n2: Node) -> Iterable[Node]:
    """Scan for all nodes nand-able with both e1 and e2"""
    interest = n1.digits | n2.digits  # max=2

    for vizone in allvisible(n1.zone, n2.zone):  # max=2
        for cell in filter(lambda c: c.digits & interest, draftborhood(board, vizone)):  # max=18
            for dig in interest:  # max=36
                node = Node.at(cell, dig)
                if node == n1 or node == n2:
                    continue
                if check_soft(node, n1) and check_soft(node, n2):
                    yield node

#### loop ALC

ALC with connected edges: `X ~ hard ~ ... ~ soft ~ X`

Rule: invalidate all draft visible from each soft-link in the chain


In [ ]:
def match_loop(chain: Chain):
    """ALC loop"""
    return len(chain) > 2 and len(chain) % 2 == 0 and chain.is_loop


def resolve_loop(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible for each soft link"""
    anchors: set[Node] = chain.anchors()  # to exclude linking back to chain

    for link in filter(lambda lnk: isinstance(lnk, SLink), chain):
        t1, t2 = link
        spoilers = set(search_soft(board, t1, t2)) - anchors
        if len(spoilers):
            yield Resolution(
                castaways={Cell(n.loc(), n.digits) for n in spoilers},
                highlights={"anchors": {t1, t2}, "chain": chain},
            )

#### open ALC

ALC with hard links at its edges: `X ~ hard ~ ... ~ hard ~ Y`

Rule: invalidate all drafts visible from both edges of such chain


In [ ]:
def match_rope(chain: Chain):
    """ALC with matching edges"""
    e1, e2 = chain.edges
    return len(chain) > 2 and len(chain) % 2 == 1 and e1.dig == e2.dig


def resolve_rope(board: Board, chain: Chain) -> Resolving:
    """Cleanup all spoilers visible from both edges"""
    anchors = chain.anchors()
    e1, e2 = chain.edges

    spoilers = set(search_soft(board, e1, e2)) - anchors
    if len(spoilers):
        yield Resolution(
            castaways={Cell(n.loc(), n.digits) for n in spoilers},
            highlights={"anchors": {e1, e2}, "chain": chain},
        )

### Search for chains

- searching for all hard inks first
- trying to connect them into chains


In [ ]:
from searching import search_breadth

In [ ]:
def expand_alc(current: Chain, links: Iterable[HLink]) -> Iterable[Chain]:
    """Expand chain to one of other hard links in the pool"""

    def close(chain):
        e1, e2 = chain.edges
        if len(chain) > 2 and isinstance(chain[0], HLink) and isinstance(chain[-1], HLink) and check_soft(e1, e2):
            yield Chain.extend(chain, SLink((e2, e1)))

    def stretch(chain, link):
        e1, e2 = chain.edges
        x1, x2 = link
        if check_soft(e2, x1):
            yield Chain.extend(chain, SLink((e2, x1)), link)
        if check_soft(e2, x2):
            yield Chain.extend(chain, SLink((e2, x2)), link.reversed())
        if check_soft(x2, e1):
            yield Chain.extendhead(chain, link, SLink((x2, e1)))
        if check_soft(x1, e1):
            yield Chain.extendhead(chain, link.reversed(), SLink((x1, e1)))

    anchors: set[Node] = current.anchors()

    def noncycling(lnk: Link):
        return lnk[0] not in anchors and lnk[1] not in anchors

    for link in filter(noncycling, links):
        for extended in stretch(current, link):
            yield from close(extended)  # yield closed before open for breadth-first
            yield extended

In [ ]:
def search_chains(current: Board, max_length: int = 8):
    links = set(search_hard_segms(current)) | set(search_hard_cells(current))

    counts = count_finals(current)

    def rate(link: Link):
        return max(counts[link[0].dig()], counts[link[1].dig()])

    links = sorted(links, key=rate)  # prioritize most present (least final-counted)
    init = [Chain.init(l) for l in links]

    # print(list(map(str, init)))

    def expanding(chain: Chain):
        yield from expand_alc(chain, links)

    def matching(chain: Chain):
        return match_loop(chain) or match_rope(chain)

    def canceling(chain: Chain):
        return len(chain) >= max_length

    yield from search_breadth(init, expanding, matching, canceling)

In [ ]:
def chains(current: Board) -> Resolving:
    for chain in search_chains(current):
        if match_loop(chain):
            res = tuple(resolve_loop(current, chain))
        elif match_rope(chain):
            res = tuple(resolve_rope(current, chain))
        else:
            res = None

        if not res:
            continue  # if didn't work

        yield from res
        break  # on first worked

## A puzzle


In [ ]:
from utils import bparse, fillempty

puzzle = bparse("""
....1.4..
..9....5.
.67......
8....51..
...6...7.
.........
.8.7..3..
2..9.....
5........
""")

puzzle = Board.transform(puzzle, fillempty)

In [ ]:
puzzle = await solve_silent(
    puzzle,
    open_singles,
    hidden_singles,
)

In [ ]:
puzzle = await solve_logging(
    puzzle,
    open_singles,
    hidden_singles,
    # locked_groups,
    open_mults_(2),
    hidden_mults_(2),
    open_mults_(3),
    hidden_mults_(3),
    open_mults_(4),
    hidden_mults_(4),
    open_mults_(5),
    hidden_mults_(5),
    chains,
    filtout={"open_singles", "hidden_singles"},
)

### GUI


In [ ]:
%%html
<!-- fuck vscode -->
<style>
:root {
    --jp-widgets-color: var(--vscode-editor-foreground);
    --jp-widgets-input-color: var(--vscode-editor-foreground);
    --jp-widgets-input-background-color: var(--vscode-editor-background);
    --jp-widgets-font-size: var(--vscode-editor-font-size);
}
.jupyter-widgets input {
   background-color: var(--jp-widgets-input-background-color);
}
.cell-output-ipywidget-background {
   background-color: transparent !important;
}
</style>

In [ ]:
import asyncio
from ipywidgets import widgets as w
from ipycanvas import hold_canvas
from IPython.display import display

from traitlets import HasTraits, Instance, Set, Unicode, observe, Enum, Bool, Dict
from canvas import SudokuCanvas
from utils import validate

In [ ]:
def click_future(button: w.Button) -> asyncio.Future[bool]:
    button.disabled = False
    future = asyncio.Future()

    def handle(b):
        button.on_click(handle, remove=True)
        button.disabled = True
        future.set_result(True)

    button.on_click(handle)

    return future


# TODO: make it cancellable somehow

In [ ]:
class GUI(HasTraits):
    """Meta-widget with reactive properties and awaitable buttons"""

    puzzle = Instance(Board)
    status = Enum(["INCOMPLETE", "SOLVED", "BROKEN"])
    counters: Instance[Counter[int]] = Instance(Counter)

    # highlighting stuff
    targets = Set(Instance(Cell))
    empties = Set(Instance(Node))
    anchors = Set(Instance(Node))
    links = Set(Instance(Link))

    # async running stuff
    running = Bool(False)
    paused = Bool(False)
    resolving = Unicode()
    inspecting = Dict(Bool(), Unicode(), default_value={})

    def __init__(self):
        super().__init__()
        self._canvas = SudokuCanvas()

        self._counters = {
            str(dig): w.Label(
                str(dig),
                layout=dict(width="auto", justify_content="center"),
                style=dict(text_color="black", background="var(--jp-info-color0)"),
            )
            for dig in DIGITS
        }
        self._counters["TOTAL"] = w.Label(
            "...",
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        self._status = w.Label(
            layout=dict(width="auto", justify_content="center"),
            style=dict(text_color="black", background="var(--jp-info-color0)"),
        )
        #
        self._running = w.Button(
            icon="play",
            style=dict(
                font_size="large",
                text_color="var(--jp-info-color0)",
                button_color="transparent",
            ),
            tooltip="not a button",
        )
        self._continue = w.Button(description="Continue", disabled=True, button_style="primary")
        self._continue.layout.visibility = "hidden"
        #
        self._inspecting = w.VBox([w.Label("Inspecting"), w.VBox()])
        self._inspecting.layout.visibility = "hidden"
        self._resolving = w.Label()

    def _repr_mimebundle_(self, **kwargs):
        return w.HBox(
            [
                w.VBox(
                    [w.Label("Status"), *self._counters.values(), self._status],
                    layout=dict(align_items="stretch", width="7em"),
                ),
                self._canvas,
                w.VBox([
                    self._running,
                    self._resolving,
                    self._continue,
                    self._inspecting,
                ]),
            ],
            layout=dict(justify_content="flex-start", align_items="stretch"),
        )._repr_mimebundle_(**kwargs)

    @observe("puzzle")
    def upd_puzzle(self, change):
        self._canvas.draw_board(self.puzzle)
        self._hlayers = set()
        self.status = validate(self.puzzle)
        self.counters = count_finals(self.puzzle)

    def toggle_layer(self, layer: int):
        if not self.puzzle:
            return
        if layer in self._hlayers:
            self._hlayers.remove(layer)
        else:
            self._hlayers.add(layer)
        self._canvas.draw_board(self.puzzle, self._hlayers)

    @observe("status")
    def upd_status(self, change):
        value = self.status
        self._status.value = value
        self._status.style.visibility = "visible" if value != "" else "hidden"
        if value == "SOLVED":
            self._status.style.background = "var(--jp-success-color0)"
        elif value == "BROKEN":
            self._status.style.background = "var(--jp-error-color0)"
        else:
            self._status.style.background = "var(--jp-info-color0)"

    @observe("counters")
    def upd_counter(self, change):
        counters = self.counters
        for dig, cnt in counters.items():
            w = self._counters[str(dig)]
            w.value = f"{dig}: ({cnt})"
        total = counters.total()
        w = self._counters["TOTAL"]
        w.value = f"Total: ({total})"

    @observe("targets", "anchors", "empties", "links")
    def redraw_highlights(self, change):
        # redrawing everything in proper order
        self._canvas.clear_highlights()
        with hold_canvas():
            for lnk in self.links:
                self._highlight_link(lnk, "blue")

            for node in self.empties:
                self._highlight_node(node, "pink")

            for node in self.anchors:
                if node.is_cellular:
                    self._highlight_node(node, "cyan")
                else:
                    self._highlight_group(node, "cyan")

            for cell in self.targets:
                self._highlight_cell(cell, "red")

    def reset_highlights(self):
        self._canvas.clear_highlights()
        self.targets = set()
        self.anchors = set()
        self.empties = set()
        self.links = set()

    def _highlight_node(self, node: Node, color: str):
        for loc in node.zone:
            for dig in node.digits:
                self._canvas.highlight_segment(loc, dig, color=color)

    def _highlight_cell(self, cell: Cell, color: str):
        for dig in cell.digits:
            self._canvas.highlight_segment(cell.loc, dig, color=color)

    def _highlight_link(self, lnk: Link, color: str):
        t1, t2 = lnk
        if t1.zone.is_cell and t2.zone.is_cell:
            self._canvas.highlight_link(
                t1.loc(),
                t1.dig(),
                t2.loc(),
                t2.dig(),
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )
        else:
            t1locs = tuple(iter(t1.zone))
            l1mid = Loc(
                sum(l.r for l in t1locs) // len(t1locs),
                sum(l.c for l in t1locs) // len(t1locs),
            )
            t2locs = tuple(iter(t2.zone))
            l2mid = Loc(
                sum(l.r for l in t2locs) // len(t2locs),
                sum(l.c for l in t2locs) // len(t2locs),
            )
            self._canvas.highlight_link(
                l1mid,
                t1.dig(),
                l2mid,
                t2.dig(),
                style=LINK_STYLES[lnk.__class__.__name__],
                color=color,
            )

    def _highlight_group(self, node: Node, color: str):
        locs = tuple(iter(node.zone))
        lmin = Loc(min(l.r for l in locs), min(l.c for l in locs))
        lmax = Loc(max(l.r for l in locs), max(l.c for l in locs))
        self._canvas.highlight_link(lmin, node.dig(), lmax, node.dig(), style="GROUP", color=color)
        self._highlight_node(node, color)

    @observe("running", "paused")
    def upd_running(self, change):
        self._running.icon = "play" if not self.running else "gear" if self.paused else "gear spin"
        self._running.disabled = self.running
        self._continue.layout.visibility = "visible" if self.running else "hidden"
        self._continue.disabled = not self.paused

    @observe("inspecting")
    def upd_inspecting(self, change):
        checkboxes = self._inspecting.children[1]
        for ch in checkboxes.children:
            ch.close()
        checkboxes.children = []
        if len(self.inspecting):
            checkboxes.children = [w.Checkbox(value=v, description=k, indent=False) for k, v in self.inspecting.items()]
            for ch in checkboxes.children:
                ch.observe(self.upd_inspecting_item, "value")
            self._inspecting.layout.visibility = "visible"
        else:
            self._inspecting.layout.visibility = "hidden"

    def upd_inspecting_item(self, change):
        checkbox = change["owner"]
        self.inspecting[checkbox.description] = checkbox.value

    @observe("resolving")
    def upd_resolving(self, change):
        self._resolving.value = self.resolving

    def click_continue(self) -> asyncio.Future[bool]:
        return click_future(self._continue)

    async def pause(self):
        self.paused = True
        await self.click_continue()
        self.paused = False


# GUI meta-widget


LINK_STYLES = {
    "Link": "SOLID",
    "HLink": "HARD",
    "SLink": "SOFT",
}

In [ ]:
debug_view = w.Output()
gui = GUI()

In [ ]:
async def solve_ui(initial: Board, *resolvers: Resolver, filtout: set[str] = set()):
    current = initial
    result = initial
    gui.puzzle = current
    gui.running = True
    gui.inspecting = {r.__name__: r.__name__ not in filtout for r in resolvers}

    try:
        iteration = 0
        async for resolver, resolution, result in solver(initial, *resolvers):
            iteration += 1
            # print(iteration, resolver.__name__)
            # pprint(resolution)
            if gui.inspecting[resolver.__name__]:
                gui.puzzle = current
                resolving = f"#{iteration} {resolver.__name__}: "
                if resolution.castaways:
                    resolving += f"-= {len(resolution.castaways)}"
                if resolution.finals:
                    resolving += f"== {len(resolution.finals)}"
                gui.resolving = resolving
                render_resolution(resolution)
                await gui.pause()
                clear_resolution()
                gui.puzzle = result
                await asyncio.sleep(0.2)
            current = result
            gui.resolving = f"#{iteration}"
    except Exception as e:
        # FIXME: the cancel button handler
        with debug_view:
            raise RuntimeError("Solver failed") from e

    gui.puzzle = result
    gui.running = False
    gui.inspecting = {}

    return result


def render_resolution(res: Resolution):
    with hold_canvas():
        gui.targets = res.castaways if res.castaways else set()

        gui.anchors = res.highlights.get("anchors", set())
        gui.empties = res.highlights.get("empties", set())

        if "chain" in res.highlights:
            gui.links = set(res.highlights["chain"])
        elif "links" in res.highlights:
            gui.links = res.highlights["links"]
        else:
            gui.links = set()


def clear_resolution():
    gui.reset_highlights()

In [ ]:
display(gui, debug_view)

In [ ]:
gui.puzzle = puzzle
gui.targets = set()
gui.anchors = set()
gui.links = set()

In [ ]:
searching = search_chains(puzzle, 10)


In [ ]:
chain = next(searching)
gui.links = set(chain)
gui.anchors = set(iterflat(chain))

In [ ]:
task = asyncio.create_task(
    solve_ui(
        puzzle,
        open_singles,
        hidden_singles,
        # locked_groups,
        open_mults_(2),
        hidden_mults_(2),
        open_mults_(3),
        hidden_mults_(3),
        open_mults_(4),
        hidden_mults_(4),
        open_mults_(5),
        hidden_mults_(5),
        chains,
        # gchains,
        # filtout={"open_singles", "hidden_singles"},
    )
)

In [ ]:
task

In [ ]:
task.cancel()